# Benchmark de `fit()` e ROC AUC no LightGBM e no Bankai

Abra o notebook com o kernel da `.venv` e execute as células em ordem. Antes de medir, rode
`maturin develop -r` na raiz do repositório para instalar o Bankai em release. O kernel precisa
de `ipykernel`, `lightgbm==4.7.0`, NumPy e scikit-learn. Inicie o notebook na raiz
do repositório para registrar o commit medido.

O notebook gera dados sintéticos e usa o sinal das duas primeiras colunas para definir
as probabilidades com que sorteia os rótulos binários. As duas
bibliotecas recebem os mesmos dados de treino e a mesma amostra independente de teste.

Temporizamos apenas `LGBMClassifier.fit()`, incluindo a preparação interna dos dados.
Depois das rodadas cronometradas, calculamos o ROC AUC no conjunto de teste usando
as probabilidades da classe positiva dos últimos modelos treinados.
Reinicie o kernel para aplicar a política OpenMP antes de importar as bibliotecas.

In [ ]:
import os
import subprocess
from statistics import median
from time import perf_counter

os.environ["OMP_WAIT_POLICY"] = "PASSIVE"

import lightgbm
import numpy as np
from IPython.display import Markdown, display
from sklearn.metrics import roc_auc_score

import bankai_lightgbm

if lightgbm.__version__ != "4.7.0":
    raise RuntimeError(f"Instale lightgbm==4.7.0. Versão atual: {lightgbm.__version__}")

seed = 42
rounds = 7
params = dict(n_estimators=200, n_jobs=1, num_leaves=31, random_state=seed, verbosity=-1)
train_rows = 50_000
test_rows = 200_000

def make_classification_sample(rng, n_rows):
    features = rng.normal(size=(n_rows, 20))
    signal = 2 * features[:, 0] + np.sin(features[:, 1])
    signal += rng.normal(scale=0.1, size=n_rows)
    probabilities = 1 / (1 + np.exp(-signal))
    return features, rng.binomial(1, probabilities)

train_seed, test_seed = np.random.SeedSequence(seed).spawn(2)
X_train, y_train = make_classification_sample(np.random.default_rng(train_seed), train_rows)
X_test, y_test = make_classification_sample(np.random.default_rng(test_seed), test_rows)

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Build esperado: release | checkout: {commit}")
print(
    f"Upstream: {lightgbm.__version__} | num_threads: {params['n_jobs']} | "
    f"OMP_WAIT_POLICY: {os.environ['OMP_WAIT_POLICY']}"
)
print(f"Treino: {X_train.shape} | teste: {X_test.shape}")
print(f"Árvores solicitadas: {params['n_estimators']} | rodadas: {rounds}")

In [ ]:
libraries = {"LightGBM upstream": lightgbm, "Bankai LGBM": bankai_lightgbm}

def fit_timed(library):
    model = library.LGBMClassifier(**params)
    start = perf_counter()
    model.fit(X_train, y_train)
    return (perf_counter() - start) * 1_000, model

for library in libraries.values():
    fit_timed(library)

samples = {name: [] for name in libraries}
models = {}
for _ in range(rounds):
    for name, library in libraries.items():
        elapsed_ms, models[name] = fit_timed(library)
        samples[name].append(elapsed_ms)

medians = {name: median(times) for name, times in samples.items()}
rows = ["| Biblioteca | Mediana por fit | Mínimo | Máximo |", "|---|---:|---:|---:|"]
for name, times in samples.items():
    rows.append(f"| {name} | {medians[name]:.2f} ms | {min(times):.2f} ms | {max(times):.2f} ms |")
ratio = medians["Bankai LGBM"] / medians["LightGBM upstream"]
rows.extend(["", f"**Tempo Bankai / upstream:** {ratio:.2f}×"])
display(Markdown("\n".join(rows)))

In [ ]:
auc = {}
for name, model in models.items():
    positive_probability = model.predict_proba(X_test)[:, 1]
    auc[name] = roc_auc_score(y_test, positive_probability)

rows = ["| Biblioteca | ROC AUC no teste |", "|---|---:|"]
for name, score in auc.items():
    rows.append(f"| {name} | {score:.8f} |")

auc_gap = abs(auc["Bankai LGBM"] - auc["LightGBM upstream"])
rows.extend(["", f"**Diferença absoluta de ROC AUC:** {auc_gap:.8f}"])
display(Markdown("\n".join(rows)))